- Accuracy = (TP+TN)/total = 94/100 = 94%. This one is a trap: a model that says "no" to everyone scores 90% here. That's why you need the others.
- Precision = TP/(TP+FP) = 6/8 = 75%. "When the model says yes, how often is it right?"
- Recall = TP/(TP+FN) = 6/10 = 60%. "Of the people who really said yes, how many did it catch?"
- F1 = 2·P·R/(P+R) = 2·0.75·0.6/1.35 ≈ 0.67. One number that balances precision and recall. It drops if either one is bad.
- ROC-AUC: every threshold (0.5, 0.3, 0.8…) gives a different trade-off between catching yeses and raising false alarms. The ROC curve plots all of those thresholds, and AUC is the area under the curve. 0.5 means coin flip and 1.0 means perfect. The plain-English version: pick one real "yes" person and one real "no" person at random, and AUC is the chance the model gave the yes person the higher probability.
- Common bug: ROC-AUC needs the probabilities from predict_proba, not the 0/1 predictions.

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

df = pd.read_csv("train.csv")  # load the data
df.head()
df.info()
df["y"].value_counts() # count how many yes

<class 'pandas.DataFrame'>
RangeIndex: 30891 entries, 0 to 30890
Data columns (total 22 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   row_id          30891 non-null  int64  
 1   age             30891 non-null  int64  
 2   job             30891 non-null  str    
 3   marital         30891 non-null  str    
 4   education       30891 non-null  str    
 5   default         30891 non-null  str    
 6   housing         30891 non-null  str    
 7   loan            30891 non-null  str    
 8   contact         30891 non-null  str    
 9   month           30891 non-null  str    
 10  day_of_week     30891 non-null  str    
 11  duration        30891 non-null  int64  
 12  campaign        30891 non-null  int64  
 13  pdays           30891 non-null  int64  
 14  previous        30891 non-null  int64  
 15  poutcome        30891 non-null  str    
 16  emp.var.rate    30891 non-null  float64
 17  cons.price.idx  30891 non-null  float64
 1

y
no     27411
yes     3480
Name: count, dtype: int64

In [2]:
# first go through each column and figure out whats in it.
for col in df.select_dtypes(include = ["object", "str"]).columns: # for each column check how many of each value
    print(df[col].value_counts())
    print()

job
admin.           7830
blue-collar      6891
technician       5047
services         2956
management       2184
retired          1297
entrepreneur     1111
self-employed    1085
housemaid         787
unemployed        773
student           685
unknown           245
Name: count, dtype: int64

marital
married     18644
single       8733
divorced     3458
unknown        56
Name: count, dtype: int64

education
university.degree      9215
high.school            7106
basic.9y               4522
professional.course    3960
basic.4y               3090
basic.6y               1694
unknown                1290
illiterate               14
Name: count, dtype: int64

default
no         24481
unknown     6409
yes            1
Name: count, dtype: int64

housing
yes        16209
no         13927
unknown      755
Name: count, dtype: int64

loan
no         25500
yes         4636
unknown      755
Name: count, dtype: int64

contact
cellular     19581
telephone    11310
Name: count, dtype: int64

month
may

Since there are exactly the same number of unknowns (755) for housing and loan, check if these are the same people

In [3]:
((df["housing"] == "unknown") & (df["loan"] == "unknown")).sum()
    

np.int64(755)

- housing and loan unknowns are the same 755 people, so after one-hotting, the two "unknown" columns are the same, so i kept one column and dropped the other to avoid redundancy. 
- i kept both yes/no columns because housing and loan are different facts for everyone else
- map y to 0/1 where yes = 1 and no = 0
- precision and recall measure how well the model finds the 1s, and "yes" is the rare outcome we actually care about catching

In [4]:

df["y"] = df["y"].map({"no": 0, "yes": 1})
print(df["y"].value_counts(dropna=False))


df["contact"] = df["contact"].map({"telephone": 0, "cellular": 1})
df["contact"].value_counts(dropna=False)

y
0    27411
1     3480
Name: count, dtype: int64


contact
1    19581
0    11310
Name: count, dtype: int64

In [5]:
month_order = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
print(df.groupby("month")["y"].mean().reindex(month_order))

day_order = ["mon", "tue", "wed", "thu", "fri"]
print(df.groupby("day_of_week")["y"].mean().reindex(day_order))

month
mar    0.498824
apr    0.206301
may    0.064764
jun    0.102790
jul    0.088989
aug    0.106664
sep    0.452656
oct    0.427027
nov    0.100225
dec    0.486486
Name: y, dtype: float64
day_of_week
mon    0.098889
tue    0.119888
wed    0.115296
thu    0.120183
fri    0.108777
Name: y, dtype: float64


In [6]:
df = pd.get_dummies(df, columns=["marital", "month", "job", "loan", "housing"], drop_first=True, dtype=int)
 # the drop_first=True is because by knowing everything else, you know the value of the one column bc the total of all the columns have to = 1
print(df.columns)
df.head()

Index(['row_id', 'age', 'education', 'default', 'contact', 'day_of_week',
       'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate',
       'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y',
       'marital_married', 'marital_single', 'marital_unknown', 'month_aug',
       'month_dec', 'month_jul', 'month_jun', 'month_mar', 'month_may',
       'month_nov', 'month_oct', 'month_sep', 'job_blue-collar',
       'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired',
       'job_self-employed', 'job_services', 'job_student', 'job_technician',
       'job_unemployed', 'job_unknown', 'loan_unknown', 'loan_yes',
       'housing_unknown', 'housing_yes'],
      dtype='str')


,row_id,age,education,default,contact,day_of_week,duration,campaign,pdays,previous,...,job_self-employed,job_services,job_student,job_technician,job_unemployed,job_unknown,loan_unknown,loan_yes,housing_unknown,housing_yes
0,0,58,professional.course,unknown,0,fri,28,1,999,0,...,0,0,0,0,0,0,0,0,0,0
1,1,37,university.degree,no,1,mon,176,2,999,0,...,0,0,0,0,0,0,0,1,0,1
2,2,25,high.school,no,1,wed,158,1,999,0,...,0,1,0,0,0,0,0,0,0,1
3,3,35,university.degree,no,1,mon,75,2,999,0,...,0,0,0,0,0,0,0,1,0,0
4,4,38,university.degree,no,0,tue,47,5,999,0,...,0,0,0,0,0,0,0,0,0,1


- Dropped day_of_week because the yes-rates were within about 2 points across all five days, so it gives the model very little to go on.
for month, i decided to do one-hot because it was an up-down zigzag pattern. Mar/Sep/Oct/Dec have high yes-rates but few calls, so those weights may be less trustworthy.
- Dropped the one row where default = yes because 1 yes is negligible in a dataset with 30k+ datapoints
- Since there's only no and unknown, mapped unknown = 1, no = 0, and renamed the column to default_unknown so the name matches what 1 means
- Dropped the row_id because thats just for identification and not a criteria
- Dropped duration because if duration = 0 then y = "no" but this is only known after a call is performed which also means y is known, so it can't be used if you wanna have a realistic predictive model

In [7]:
df = df.drop(columns = ["day_of_week", "row_id", "loan_unknown", "duration"]) 
df = df[df["default"] != "yes"]
df["default"] = df["default"].map({"unknown": 1, "no": 0})
df = df.rename(columns={"default": "default_unknown"})
print(df["default_unknown"].value_counts(dropna=False))


default_unknown
0    24481
1     6409
Name: count, dtype: int64


- previous
    - How many times did we call this person in the last campaign? (0 = never)
- poutcome
    - How did the last campaign go for them? (nonexistent / failure / success)
- pdays
    - How many days ago was that last contact? (999 = "no number")

In [8]:
print(pd.crosstab(df["poutcome"], df["pdays"] == 999))
print()
print(pd.crosstab(df["poutcome"], df["previous"] == 0))
print()
print(pd.crosstab(df["previous"] == 0, df["pdays"] == 999))

pdays        False  True 
poutcome                 
failure        111   3060
nonexistent      0  26690
success       1029      0

previous     False  True 
poutcome                 
failure       3171      0
nonexistent      0  26690
success       1029      0

pdays     False  True 
previous              
False      1140   3060
True          0  26690


In [9]:
df.groupby(["poutcome", df["pdays"] == 999])["y"].mean()

poutcome     pdays
failure      False    0.513514
             True     0.130065
nonexistent  True     0.088273
success      False    0.650146
Name: y, dtype: float64

- I one-hot encoded poutcome because the yes rates go from 8.8% (nonexistent) to about 14% (failure) (about 14% overall; 13% without a day count, 51% with one) to 65% (success). 
- Ordinal 0/1/2 would assume each step changes the odds by the same amount, but the jumps are very uneven
    - btw ordinal means the categories have a real order (0,1,2), and 1 shared weight

- drop poutcome_nonexistent so that the "never contacted before" is the baseline. the other two weights then show the effect of failing or succeeding last time compared to no history. nonexistent is also the most common value, and it gets rid of redundancy that appears with previous == 0.

- I also replaced pdays with a 1/0 flag (has a real day count). The yes-rate by individual day count was noisy w/ no real pattern, so it was just flagged with count or no count instead of dependent on days.

In [10]:
df["pdays_known"] = (df["pdays"] != 999).astype(int)
df = pd.get_dummies(df, columns=["poutcome"], dtype=int)
df = df.drop(columns = ["poutcome_nonexistent", "pdays"]) 
df["pdays_known"].value_counts()

pdays_known
0    29750
1     1140
Name: count, dtype: int64

EDUCATION:
- Merged basic.4y, basic.6y, and basic.9y into one level (2). Their yes-rates are all close (9.8%, 7.9%, 7.8%) and don't go up with more years, so keeping them as separate steps would just add noise
- Order: 1 = illiterate, 2 = basic, 3 = high school, 4 = professional course, 5 = university degree. The yes-rate goes up roughly steadily from basic (~8%) to high school (10.8%) to professional course (11.6%) to university (13.9%), so ordinal is good
- Kept illiterate at 1 - logical bottom of the scale. its 28.6% yes-rate is based on only 14 people, so it's too small to affect the model much
- Gave unknown its own 1/0 flag (education_unknown) instead of guessing a level, because we don't actually know their education. the education column still needs a number for those rows, so I filled them with the median, 3 (high school). The exact value matters less because the flag lets the model adjust for those rows

In [11]:
df.groupby("education")["y"].agg(["mean", "count"])


,mean,count
education,,
basic.4y,0.098058,3090
basic.6y,0.079103,1694
basic.9y,0.078284,4522
high.school,0.107515,7106
illiterate,0.285714,14
professional.course,0.116444,3959
university.degree,0.139121,9215
unknown,0.137984,1290


In [12]:
df["education_unknown"] = (df["education"] == "unknown").astype(int)
df["education"] = df["education"].map({
    "illiterate": 1,
    "basic.4y": 2,
    "basic.6y": 2,
    "basic.9y": 2,
    "high.school": 3,
    "professional.course": 4,
    "university.degree": 5,
})
edu_median = df["education"].median()
print(edu_median)
df["education"] = df["education"].fillna(edu_median)
df["education"].value_counts(dropna=False)

3.0


education
2.0    9306
5.0    9215
3.0    8396
4.0    3959
1.0      14
Name: count, dtype: int64

In [13]:
df.select_dtypes(include=["object", "str"]).columns

Index([], dtype='str')

MODEL TIMEEEE

In [14]:
X = df.drop(columns=["y"])
y = df["y"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,       # 20% for testing
    random_state = 1000,   # any random number, makes the split the same every run
    stratify=y,          # keep the same yes/no ratio in both piles
)
print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())
#LogisticRegression().fit()


(24712, 41) (6178, 41)
0.11265781806409841 0.11265781806409841


In [15]:
num_cols = ["age", "campaign", "previous", "emp.var.rate", "cons.price.idx",
            "cons.conf.idx", "euribor3m", "nr.employed"]  # add "education" if you decide to scale it
X_train[num_cols].describe()


,age,campaign,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,24712.000000,24712.000000,24712.000000,24712.000000,24712.000000,24712.000000,24712.000000,24712.000000
mean,39.986444,2.568226,0.171131,0.078521,93.572905,-40.473875,3.621276,5167.020282
std,10.424448,2.786764,0.490915,1.574435,0.578745,4.638534,1.734400,72.073142
min,17.000000,1.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.600000
25%,32.000000,1.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.100000
50%,38.000000,2.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.000000,3.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,98.000000,56.000000,6.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


SCALING:
- StandardScaler: z = (x − mean) / std. Every column ends up with mean 0 and std 1. A value means "how many standard deviations from average."
- MinMaxScaler: (x − min) / (max − min). Squeezes every column into 0 to 1, where the smallest value becomes 0 and the largest becomes 1.

- Used StandardScaler for age, campaign, and previous. when doing describe, these have extreme values (campaign max is 56 when the median is 2, and age max is 98). With MinMax, that one extreme value would become 1 and squash almost everyone else close to 0 (most people's campaign would land between 0 and 0.04). StandardScaler is less dominated by a single max.
- Used MinMaxScaler for the 5 economic columns and education. The economic columns stay inside a fixed, narrow range set by the economy, with no extreme outliers. Education is a fixed 1–5 scale, so MinMax maps it evenly to 0, 0.25, 0.5, 0.75, 1 and keeps the equal steps.
- Fit both scalers on the training set only, then applied them to the test set, so no test info leaks into training.
- Left the 0/1 columns (one-hot columns, contact, default_unknown, pdays_known, education_unknown) unscaled, because they're already 0 or 1.

In [16]:
std_cols = ["campaign",  "age", "previous"]
mm_cols  = ["emp.var.rate", "cons.price.idx",
            "cons.conf.idx", "euribor3m", "nr.employed", "education"]

std_scaler = StandardScaler()
std_scaler.fit(X_train[std_cols])                              # learn from TRAIN only
X_train[std_cols] = std_scaler.transform(X_train[std_cols])    # apply to train
X_test[std_cols]  = std_scaler.transform(X_test[std_cols])     # apply to test (no fit!)

mm_scaler = MinMaxScaler()
mm_scaler.fit(X_train[mm_cols])                              # learn from TRAIN only
X_train[mm_cols] = mm_scaler.transform(X_train[mm_cols])    # apply to train
X_test[mm_cols]  = mm_scaler.transform(X_test[mm_cols])     # apply to test (no fit!)

print(X_train[std_cols].describe())
print(X_train[mm_cols].describe())

           campaign           age      previous
count  2.471200e+04  2.471200e+04  2.471200e+04
mean   9.200942e-18 -1.621666e-16  1.293882e-17
std    1.000020e+00  1.000020e+00  1.000020e+00
min   -5.627522e-01 -2.205096e+00 -3.486041e-01
25%   -5.627522e-01 -7.661418e-01 -3.486041e-01
50%   -2.039058e-01 -1.905601e-01 -3.486041e-01
75%    1.549406e-01  6.728124e-01 -3.486041e-01
max    1.917380e+01  5.565257e+00  1.187372e+01
       emp.var.rate  cons.price.idx  cons.conf.idx     euribor3m  \
count  24712.000000    24712.000000   24712.000000  24712.000000   
mean       0.724692        0.534647       0.432055      0.677233   
std        0.328007        0.225544       0.194081      0.393199   
min        0.000000        0.000000       0.000000      0.000000   
25%        0.333333        0.340608       0.338912      0.160961   
50%        0.937500        0.603274       0.376569      0.957379   
75%        1.000000        0.698753       0.602510      0.980957   
max        1.000000     

In [17]:
model = LogisticRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)               # 0/1 guesses (cutoff 0.5)
y_prob = model.predict_proba(X_test)[:, 1]   # probability of yes
print(y_pred[:10])
print(y_prob[:10])

[0 0 0 0 0 0 0 0 0 0]
[0.0572024  0.05000239 0.03331727 0.07434571 0.0500584  0.35906858
 0.05310835 0.07884886 0.08529164 0.05444027]


In [18]:
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:   ", recall_score(y_test, y_pred))
print("F1:       ", f1_score(y_test, y_pred))
print("ROC-AUC:  ", roc_auc_score(y_test, y_prob))   # probabilities, not y_pred

print(confusion_matrix(y_test, y_pred))

Accuracy:  0.8973777921657494
Precision: 0.6336206896551724
Recall:    0.21120689655172414
F1:        0.3168103448275862
ROC-AUC:   0.7980071666100551
[[5397   85]
 [ 549  147]]


In [19]:
final_X = pd.read_csv("test_features.csv")
final_labels = pd.read_csv("test_labels.csv")
print(final_X.shape, final_labels.shape)
print(final_labels.columns)

(10297, 21) (10297, 2)
Index(['row_id', 'y'], dtype='str')


In [ ]:
# ===== FINAL TEST: apply the SAME preprocessing as training, fit NOTHING =====
# Rule: only .transform / .predict here. Never .fit, so no test info leaks into the model.

# --- 1. Line up answers with rows ---
# features and labels are separate files, so join them on row_id
# so each person's answer stays attached to the right row
final = final_X.merge(final_labels, on="row_id")
final_y = final["y"].map({"no": 0, "yes": 1})   # same 0/1 target as training (yes = 1)
f = final.drop(columns=["y"])                    # f = features only

# --- 2. Same drops as training ---
# row_id: just an ID | day_of_week: yes-rate flat across days | duration: leakage (only known after the call)
# NOTE: no rows dropped here (unlike the default = yes row in training), because every test row gets graded
f = f.drop(columns=["row_id", "day_of_week", "duration"])

# --- 3. Same single 0/1 columns ---
f["contact"] = f["contact"].map({"telephone": 0, "cellular": 1})   # only 2 values, so 1 column is enough
# compare instead of .map: a "yes" in default would become NaN with .map and crash the model.
# This way unknown = 1, anything else (no or yes) = 0
f["default_unknown"] = (f["default"] == "unknown").astype(int)
# 999 is a fake "no day count" number. The flag keeps the signal (has a real day count vs. not)
f["pdays_known"] = (f["pdays"] != 999).astype(int)
f = f.drop(columns=["default", "pdays"])        # originals replaced by the new columns

# --- 4. Education: same ordinal order, same unknown flag ---
f["education_unknown"] = (f["education"] == "unknown").astype(int)   # flag BEFORE mapping, or unknown gets lost
f["education"] = f["education"].map({
    "illiterate": 1, "basic.4y": 2, "basic.6y": 2, "basic.9y": 2,   # basic levels merged (similar yes-rates)
    "high.school": 3, "professional.course": 4, "university.degree": 5,
})
# unknown isn't in the map, so it became NaN. Fill it with the TRAINING median (3),
# not a median computed from the test set
f["education"] = f["education"].fillna(edu_median)

# --- 5. One-hot, then force the exact training columns ---
# no drop_first here: it drops whatever comes first alphabetically IN THIS FILE,
# which could differ from training if a category is missing
f = pd.get_dummies(f, columns=["marital", "month", "job", "loan", "housing", "poutcome"], dtype=int)
# reindex = keep exactly the 41 training columns, in the same order:
#   - baselines dropped in training (divorced, apr, admin., no, nonexistent, loan_unknown) disappear again
#   - a category the test set doesn't have gets a column of 0s
#   - a category the model never saw (e.g. a new month) is thrown away, so the row is treated as the baseline
f = f.reindex(columns=X_train.columns, fill_value=0)

# --- 6. Scale with the TRAINING scalers (transform only, NO fit) ---
# uses the mean/std and min/max learned from the training set
f[std_cols] = std_scaler.transform(f[std_cols])   # age, campaign, previous (have extreme values)
f[mm_cols] = mm_scaler.transform(f[mm_cols])      # economic columns + education (fixed ranges)
# MinMax values can land slightly outside 0–1 on new data. That's fine and expected

# --- 7. Sanity check ---
# expect (10297, 41) 0 0 → 41 columns like training, no blanks in features or labels
print(f.shape, f.isna().sum().sum(), final_y.isna().sum())

(10297, 41) 0 0


In [21]:
final_pred = model.predict(f)
final_prob = model.predict_proba(f)[:, 1]

print("Accuracy: ", accuracy_score(final_y, final_pred))
print("Precision:", precision_score(final_y, final_pred))
print("Recall:   ", recall_score(final_y, final_pred))
print("F1:       ", f1_score(final_y, final_pred))
print("ROC-AUC:  ", roc_auc_score(final_y, final_prob))
print(confusion_matrix(final_y, final_pred))

Accuracy:  0.9011362532776537
Precision: 0.6792929292929293
Recall:    0.23189655172413792
F1:        0.34575835475578404
ROC-AUC:   0.7858739852739713
[[9010  127]
 [ 891  269]]
